# **Laboratorio #5**

*Juan Diego Letona*
*20230285*

In [5]:
#preparamos el corpus para trabajar secuencias

from pathlib import Path
import re
import random

TXT_PATH = Path("don-quijote.txt")

if not TXT_PATH.exists():
    raise FileNotFoundError(f"No se encontro el archivo {TXT_PATH}")

#leemos el archivo aunque venga con otra codificacion
def leer_texto(path):
    for encoding in ["utf-8", "utf-8-sig", "latin-1", "cp1252"]:
        try:
            return path.read_text(encoding=encoding)
        except UnicodeDecodeError:
            pass
    raise UnicodeDecodeError("No se pudo leer el archivo")

texto = leer_texto(TXT_PATH)
texto = re.sub(r"\s+", " ", texto).strip()

#hacemos la segmentacion en oraciones
oraciones_raw = re.split(r"(?<=[.!?])\s+", texto)

#tokenizamos cada oracion en palabras sin quitar stopwords ni lematizar
patron_palabras = re.compile(
    r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]+(?:[-'][A-Za-zÁÉÍÓÚÜÑáéíóúüñ]+)*|\d+(?:[.,]\d+)*",
    re.UNICODE
)

oraciones_tokenizadas = [
    patron_palabras.findall(oracion)
    for oracion in oraciones_raw
]

oraciones_tokenizadas = [
    oracion
    for oracion in oraciones_tokenizadas
    if len(oracion) > 0
]

#revisamos que la carga y tokenizacion tengan sentido
print(f"Archivo leido {TXT_PATH}")
print(f"Total de caracteres {len(texto):,}")
print(f"Total de oraciones tokenizadas {len(oraciones_tokenizadas):,}")
print("Ejemplo de oracion tokenizada")
print(oraciones_tokenizadas[0][:40])

Archivo leido don-quijote.txt
Total de caracteres 2,107,992
Total de oraciones tokenizadas 9,577
Ejemplo de oracion tokenizada
['The', 'Project', 'Gutenberg', 'EBook', 'of', 'Don', 'Quijote', 'by', 'Miguel', 'de', 'Cervantes', 'Saavedra', 'This', 'eBook', 'is', 'for', 'the', 'use', 'of', 'anyone', 'anywhere', 'at', 'no', 'cost', 'and', 'with', 'almost', 'no', 'restrictions', 'whatsoever']


In [6]:
#agregamos tokens especiales al inicio y al final de cada oracion

TOKEN_INICIO = "~~"
TOKEN_FIN = "~~"

oraciones_con_tokens = [
    [TOKEN_INICIO] + oracion + [TOKEN_FIN]
    for oracion in oraciones_tokenizadas
]

print(f"Total de oraciones con tokens especiales {len(oraciones_con_tokens):,}")
print("Ejemplo con tokens de inicio y fin")
print(oraciones_con_tokens[0][:45])

Total de oraciones con tokens especiales 9,577
Ejemplo con tokens de inicio y fin
['~~', 'The', 'Project', 'Gutenberg', 'EBook', 'of', 'Don', 'Quijote', 'by', 'Miguel', 'de', 'Cervantes', 'Saavedra', 'This', 'eBook', 'is', 'for', 'the', 'use', 'of', 'anyone', 'anywhere', 'at', 'no', 'cost', 'and', 'with', 'almost', 'no', 'restrictions', 'whatsoever', '~~']


In [7]:
#dividimos las oraciones en entrenamiento validacion y prueba

SEED = 42
random.seed(SEED)

oraciones = oraciones_con_tokens.copy()
random.shuffle(oraciones)

total_oraciones = len(oraciones)

n_train = int(total_oraciones * 0.80)
n_val = int(total_oraciones * 0.10)

train = oraciones[:n_train]
val = oraciones[n_train:n_train + n_val]
test = oraciones[n_train + n_val:]

print(f"Total de oraciones {total_oraciones:,}")
print(f"Entrenamiento {len(train):,} ({len(train) / total_oraciones:.1%})")
print(f"Validacion {len(val):,} ({len(val) / total_oraciones:.1%})")
print(f"Prueba {len(test):,} ({len(test) / total_oraciones:.1%})")

Total de oraciones 9,577
Entrenamiento 7,661 (80.0%)
Validacion 957 (10.0%)
Prueba 959 (10.0%)


In [8]:
#calculamos el vocabulario de entrenamiento y las palabras no vistas en prueba

tokens_especiales = {TOKEN_INICIO, TOKEN_FIN}

palabras_train = [
    token
    for oracion in train
    for token in oracion
    if token not in tokens_especiales
]

palabras_test = [
    token
    for oracion in test
    for token in oracion
    if token not in tokens_especiales
]

vocabulario_train = set(palabras_train)

palabras_oov = [
    palabra
    for palabra in palabras_test
    if palabra not in vocabulario_train
]

vocab_size = len(vocabulario_train)
proporcion_oov = len(palabras_oov) / len(palabras_test)

#reportamos las metricas pedidas
print(f"Tamano del vocabulario de entrenamiento {vocab_size:,}")
print(f"Total de palabras en prueba {len(palabras_test):,}")
print(f"Palabras de prueba no vistas en entrenamiento {len(palabras_oov):,}")
print(f"Proporcion OOV en prueba {proporcion_oov:.4f}")
print(f"Proporcion OOV en prueba {proporcion_oov:.2%}")

print("\nEjemplos de palabras OOV")
print(sorted(set(palabras_oov))[:50])

Tamano del vocabulario de entrenamiento 22,511
Total de palabras en prueba 38,462
Palabras de prueba no vistas en entrenamiento 1,514
Proporcion OOV en prueba 0.0394
Proporcion OOV en prueba 3.94%

Ejemplos de palabras OOV
['10', '1500', '1887', '596', '60', '801', '809', '84116', 'ACADÉMICOS', 'Abre', 'Acomodada', 'Acudid', 'Aderezáronse', 'Alabo', 'Alcaná', 'Alcocer', 'Alejandría', 'Algo', 'Amohinábase', 'Amohinóse', 'Any', 'Apagaron', 'Apenino', 'Aprieta', 'Asilde', 'Augusta', 'Babie', 'Basilea', 'Bañares', 'Benalcázar', 'Bonita', 'Burguillos', 'BÉJAR', 'CONDE', 'Callaban', 'Cansábanse', 'Capilla', 'Carloto', 'Caño', 'Charní', 'Ciertos', 'City', 'Cogiéronle', 'Comendador', 'Compluto', 'Concilio', 'Confusas', 'Creating', 'Creyóle', 'Cuitada']


El vocabulario del conjunto de entrenamiento tiene 22,511 palabras distintas.
Sin embargo, en el conjunto de prueba aparecio una proporcion OOV de 3.94%.
Esto significa que algunas palabras de prueba nunca aparecieron durante el entrenamiento.

Este problema se relaciona directamente con la dispersion de datos, o data sparsity,
porque aunque el corpus tenga bastante texto, no todas las palabras ni todas las
combinaciones posibles aparecen suficientes veces. Como el modelo trabaja palabra
por palabra, necesita ejemplos previos para aprender buenas probabilidades.

Cuando aparece una palabra nueva o una secuencia rara en prueba, el modelo no tiene
suficiente evidencia para estimarla bien. Esa falta de ejemplos es la idea central
de la data sparsity. Mientras mas pequeno o mas variado sea el corpus, mas probable
es encontrar palabras nunca vistas en validacion o prueba.